Import section

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append("../examples")
from generate_data import generate_data
sys.path.append("../lib")
from sPOD_algo import (
    shifted_POD,
    sPOD_Param,
    give_interpolation_error,
)
from sPOD_algo_old_tv import( 
    shifted_POD_old ,
)
from transforms import Transform

Choose case and method

In [ ]:
PIC_DIR = "../images/"
SAVE_FIG = True
PLOT_VT = False

CASE = "crossing_waves"
#CASE = "sine_waves"
#CASE = "sine_waves_noise"
#CASE = "multiple_ranks"

Nx = 200        # number of grid points in x
Nt = Nx // 2    # number of time intervals
Niter = 100      # number of sPOD iterations

VARIANT = "JFBTV"
#VARIANT = "BFBTV"
#VARIANT = "ALMTV"
#VARIANT = "JFBTV_megaframe"
#VARIANT = "ALMTV_megaframe"

Generate data

In [ ]:
plt.close("all")
fields, shift_list, nmodes_exact, L, dx = generate_data(Nx, Nt, CASE)
data_shape = [Nx, 1, 1, Nt]
qmat = np.reshape(fields, [Nx, Nt])

Create transforms

In [ ]:
transfos = [
    Transform(data_shape, [L], shifts=shift_list[0], dx=[dx], interp_order=5),
    Transform(data_shape, [L], shifts=shift_list[1], dx=[dx], interp_order=5),
]

interp_err = np.max([give_interpolation_error(fields, transfo) for transfo in transfos])
print("interpolation error: {:1.2e}".format(interp_err))

Define some other parameters for FB and ALM (ugly code, I might change those later)

In [ ]:
mu0 = Nx * Nt / (4 * np.sum(np.abs(qmat))) 
myparams = sPOD_Param()
myparams.lambda_s = 0.3    #1 / np.sqrt(np.maximum(Nx, Nt))
myparams.lambda_E = 0.0135
myparams.maxit = Niter
param_alm = mu0/10
nmodes = None

# error matrix makes sense only for cases with noise and/or non-zero interpolation error
myparams.isError = False
if (VARIANT != "J2") and ((CASE == "sine_waves_noise") or (CASE == "sine_waves")):
    myparams.isError = True

myparams.tv_niter  = 1
myparams.tv_mu     = 0.01       # 0.01 is probably the best -- 
                                # too small values mean no smoothing, too big values worsen the result (higher ranks, higher errors)

if VARIANT == "ALMTV":
    #param_alm = mu0  # adjust for case
    #myparams.lambda_s = 1
    if CASE == "multiple_ranks":
        myparams.lambda_s = 1
        param_alm = mu0*0.1
    if CASE == "sine_waves_noise":
        myparams.lambda_s = 1
        myparams.lambda_E = 0.05
        param_alm = mu0

if VARIANT == "ALMTV_megaframe":
    myparams.lambda_s = 4
    if CASE == "sine_waves_noise":
        myparams.lambda_E = 0.1

elif VARIANT == "JFBTV":
    myparams.lambda_s = 1
    if CASE == "crossing_waves":
        myparams.lambda_s = 0.05
    if CASE == "multiple_ranks":
        myparams.lambda_s = 0.1486
    if CASE == "sine_waves":
        #myparams.lambda_s = 0.35
        myparams.lambda_s = 0.3
        myparams.lambda_E = 0.0135
    if CASE == "sine_waves_noise":
        myparams.lambda_s = 0.3
        myparams.lambda_E = 0.0135

elif VARIANT == "BFBTV":
    myparams.lambda_s = 1
    if CASE == "crossing_waves":
        myparams.lambda_s = 0.05
    if CASE == "multiple_ranks":
        myparams.lambda_s = 0.1486
    if CASE == "sine_waves":
        #myparams.lambda_s = 0.35
        myparams.lambda_s = 0.3
        myparams.lambda_E = 0.0135
    if CASE == "sine_waves_noise":
        myparams.lambda_s = 0.3
        myparams.lambda_E = 0.0135
 

Run the proposed TV variant 1 (Arthur's approach)

In [ ]:
METHOD = VARIANT
ret_tv1 = shifted_POD(qmat, transfos, myparams, METHOD, param_alm, nmodes=nmodes)
sPOD_frames_tv1, qtilde_tv1, rel_err_tv1 = ret_tv1.frames, ret_tv1.data_approx, ret_tv1.rel_err_hist
rank_tv1 = ret_tv1.ranks
print()
print()

Run variant without TV

In [ ]:
METHOD = VARIANT.replace("TV","")
myparams.tv_niter = 0

ret = shifted_POD(qmat, transfos, myparams, METHOD, param_alm, nmodes=nmodes)
sPOD_frames, qtilde, rel_err = ret.frames, ret.data_approx, ret.rel_err_hist
rank = ret.ranks
print()
print()

Run older TV variant (Philipp's thesis version) 

I already deleted the tv iterations from normal sPOD algorithms -- I load the algorithm from a different file

In [ ]:
METHOD = VARIANT.replace("TV","")
myparams.tv_niter = 40
myparams.tv_mu = 0.01

ret_old = shifted_POD_old(qmat, transfos, myparams, METHOD, param_alm, nmodes=nmodes)
sPOD_frames_old, qtilde_old, rel_err_old = ret_old.frames, ret_old.data_approx, ret_old.rel_err_hist
rank_old = ret_old.ranks
print()
print()

Plot results visualisation

In [ ]:
# if we have an error matrix, there will be a window for the error matrix
if (myparams.isError == True):  
    gridspec = {"width_ratios": [1, 1, 1, 1, 1]}
    fig, ax = plt.subplots(3, 5, figsize=(14, 7), gridspec_kw=gridspec, num=101, layout='constrained')
else:
    gridspec = {"width_ratios": [1, 1, 1, 1]}
    fig, ax = plt.subplots(3, 4, figsize=(12, 7), gridspec_kw=gridspec, num=101, layout='constrained')    
mycmap = "viridis"
vmin = np.min(qmat) * 0.6
vmax = np.max(qmat) * 0.6

### original field
ax[0,0].pcolormesh(qmat, vmin=vmin, vmax=vmax, cmap=mycmap)
ax[0,0].set_title(r"$\mathbf{Q}$")
ax[0,0].axis("off")

### qtilde from algorithm without TV
ax[0,1].pcolormesh(qtilde, vmin=vmin, vmax=vmax, cmap=mycmap)
ax[0,1].set_title(r"$\tilde{\mathbf{Q}}$ -- without TV -- ranks " + str(ret.ranks))
ax[0,1].axis("off")

### frames from algorithm without TV
k_frame = 0
ax[0,2].pcolormesh(sPOD_frames[k_frame].build_field(), vmin=vmin, vmax=vmax, cmap=mycmap)
ax[0,2].set_title(r"$\hat{\mathbf{Q}}^" + str(k_frame + 1) + "$")
ax[0,2].axis("off")

k_frame = 1
im3 = ax[0,3].pcolormesh(sPOD_frames[k_frame].build_field(), vmin=vmin, vmax=vmax, cmap=mycmap)
ax[0,3].set_title(r"$\hat{\mathbf{Q}}^" + str(k_frame + 1) + "$")
ax[0,3].axis("off")

if myparams.isError == True:
    # extra plotting error matrix
    im4 = ax[0,4].pcolormesh(ret.error_matrix, vmin=vmin, vmax=vmax, cmap=mycmap)
    ax[0,4].set_title(r"$\mathbf{E}$")
    ax[0,4].axis("off")

### colorbar
if myparams.isError == True:
    plt.colorbar(im4, ax=ax[0, 4], location='right', shrink=0.8)
else:
    plt.colorbar(im3, ax=ax[0, 3], location='right', shrink=0.8)

############## new TV algorithm #######################
ax[1,0].axis("off")

### qtilde
ax[1,1].pcolormesh(qtilde_tv1, vmin=vmin, vmax=vmax, cmap=mycmap)
ax[1,1].set_title(r"$\tilde{\mathbf{Q}}$ -- new TV -- ranks " + str(ret.ranks))
ax[1,1].axis("off")

### frames
k_frame = 0
ax[1,2].pcolormesh(sPOD_frames_tv1[k_frame].build_field(), vmin=vmin, vmax=vmax, cmap=mycmap)
ax[1,2].set_title(r"$\mathbf{Q}^" + str(k_frame + 1) + "$")
ax[1,2].axis("off")

k_frame = 1
ax[1,3].pcolormesh(sPOD_frames_tv1[k_frame].build_field(), vmin=vmin, vmax=vmax, cmap=mycmap)
ax[1,3].set_title(r"$\mathbf{Q}^" + str(k_frame + 1) + "$")
ax[1,3].axis("off")

if (myparams.isError == True):
    # extra plotting error matrix
    ax[1,4].pcolormesh(ret_tv1.error_matrix, vmin=vmin, vmax=vmax, cmap=mycmap)
    ax[1,4].set_title(r"$\mathbf{E}$")
    ax[1,4].axis("off")


############## old TV algorithm #######################
ax[2,0].axis("off")

### qtilde
ax[2,1].pcolormesh(qtilde_old, vmin=vmin, vmax=vmax, cmap=mycmap)
ax[2,1].set_title(r"$\tilde{\mathbf{Q}}$ -- old TV -- ranks " + str(ret_old.ranks))
ax[2,1].axis("off")

### frames
k_frame = 0
ax[2,2].pcolormesh(sPOD_frames_old[k_frame].build_field(), vmin=vmin, vmax=vmax, cmap=mycmap)
ax[2,2].set_title(r"$\mathbf{Q}^" + str(k_frame + 1) + "$")
ax[2,2].axis("off")

k_frame = 1
ax[2,3].pcolormesh(sPOD_frames_old[k_frame].build_field(), vmin=vmin, vmax=vmax, cmap=mycmap)
ax[2,3].set_title(r"$\mathbf{Q}^" + str(k_frame + 1) + "$")
ax[2,3].axis("off")

if (myparams.isError == True):
    # extra plotting error matrix
    ax[2,4].pcolormesh(ret_old.error_matrix, vmin=vmin, vmax=vmax, cmap=mycmap)
    ax[2,4].set_title(r"$\mathbf{E}$")
    ax[2,4].axis("off")

for row in range(3):
    for axes in ax[row, :]:
        axes.set_aspect(0.4)

if SAVE_FIG:
    plt.savefig(PIC_DIR + "01_pictures_%s_%s_%d.png"%(CASE, VARIANT, myparams.tv_niter))
plt.show()

Plot convergence

In [ ]:
plt.close(11)
xlims = [-1, Niter]
ylims = [0, max(np.max(ret.ranks_hist[:]), np.max(ret_tv1.ranks_hist[:]), np.max(ret_old.ranks_hist[:]))+1]     # we want ylims the same for all plots
ylims2 = [min(min(rel_err), min(rel_err_tv1), min(rel_err_old)), 1]

fig, ax = plt.subplots(1, 3, figsize=(15, 4), num=11)

# without TV #############################
ax[0].set_title("%s without TV"%(METHOD))

if ("megaframe" not in METHOD):
    ax[0].plot(ret.ranks_hist[0], "+", color='tab:blue', label="$\mathrm{rank}(\mathbf{Q}^1)$")
    ax[0].plot(ret.ranks_hist[1], "x", color='tab:cyan', label="$\mathrm{rank}(\mathbf{Q}^2)$")
else:
    ax[0].plot(ret.ranks_hist, "+", color="tab:blue", label="$\mathrm{rank}(\mathbf{\hat{Q}})$")
ax[0].plot(xlims, [nmodes_exact[0], nmodes_exact[0]], "k--", label="exact rank $r_1=%d$" % nmodes_exact[0])
ax[0].plot(xlims, [nmodes_exact[1], nmodes_exact[1]], "k-", label="exact rank $r_2=%d$" % nmodes_exact[1])
ax[0].set_xlim(xlims)
ax[0].set_ylim(ylims)
ax[0].set_xlabel("iterations")
ax[0].set_ylabel("rank $r_k$", color='tab:blue')
ax[0].tick_params(axis='y', labelcolor='tab:blue')
ax[0].legend()

ax0 = ax[0].twinx()
ax0.plot(rel_err, color='tab:orange')
ax0.set_ylabel('Rel. error', color='tab:orange')
ax0.set_ylim(ylims2)
ax0.set_yscale('log')
ax0.tick_params(axis='y', labelcolor='tab:orange')

# Arthur's TV ##########################
ax[1].set_title("%s new TV"%(METHOD))
if ("megaframe" not in METHOD):
    ax[1].plot(ret_tv1.ranks_hist[0], "+", color='tab:blue', label="$\mathrm{rank}(\mathbf{Q}^1)$")
    ax[1].plot(ret_tv1.ranks_hist[1], "x", color='tab:cyan', label="$\mathrm{rank}(\mathbf{Q}^2)$")
else:
    ax[1].plot(ret_tv1.ranks_hist, "+", color="tab:blue", label="$\mathrm{rank}(\mathbf{\hat{Q}})$")
ax[1].plot(xlims, [nmodes_exact[0], nmodes_exact[0]], "k--", label="exact rank $r=%d$" %max(nmodes_exact))
ax[1].set_xlim(xlims)
ax[1].set_ylim(ylims)
ax[1].set_xlabel("iterations")
ax[1].set_ylabel("rank $r_k$", color='tab:blue')
ax[1].tick_params(axis='y', labelcolor='tab:blue')
ax[1].legend()

ax1 = ax[1].twinx()
ax1.plot(rel_err_tv1, color='tab:orange')
ax1.set_ylabel('Rel. error', color='tab:orange')
ax1.set_yscale('log')
ax1.set_ylim(ylims2)
ax1.tick_params(axis='y', labelcolor='tab:orange')

# Philipp's TV ##########################
ax[2].set_title("%s old TV"%(METHOD))
if ("megaframe" not in METHOD):
    ax[2].plot(ret_old.ranks_hist[0], "+", color='tab:blue', label="$\mathrm{rank}(\mathbf{Q}^1)$")
    ax[2].plot(ret_old.ranks_hist[1], "x", color='tab:cyan', label="$\mathrm{rank}(\mathbf{Q}^2)$")
else:
    ax[2].plot(ret_old.ranks_hist, "+", color="tab:blue", label="$\mathrm{rank}(\mathbf{\hat{Q}})$")
ax[2].plot(xlims, [nmodes_exact[0], nmodes_exact[0]], "k--", label="exact rank $r=%d$" %max(nmodes_exact))
ax[2].set_xlim(xlims)
ax[2].set_ylim(ylims)
ax[2].set_xlabel("iterations")
ax[2].set_ylabel("rank $r_k$", color='tab:blue')
ax[2].tick_params(axis='y', labelcolor='tab:blue')
ax[2].legend()

ax2 = ax[2].twinx()
ax2.plot(rel_err_old, color='tab:orange')
ax2.set_ylabel('Rel. error', color='tab:orange')
ax2.set_yscale('log')
ax2.set_ylim(ylims2)
ax2.tick_params(axis='y', labelcolor='tab:orange')

plt.tight_layout()

if SAVE_FIG:
    plt.savefig(PIC_DIR + "01_convergence_%s_%s_%d.png"%(CASE, VARIANT, myparams.tv_niter))
plt.show()